# LoRaWAN: sensor yang jauhnya berkilo-kilometer

Satu frame di udara, join over-the-air secara manual, anggaran airtime, dan satu jaringan utuh — gateway yang berbicara protokol Semtech UDP ke network server ringan — berjalan di memori.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.18.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.18.0-preview.1"

## Satu frame di udara
Contoh klasik: 17 byte, payload dienkripsi dengan AppSKey dan ditandatangani dengan NwkSKey.

In [ ]:
using IoTCom.Net.Protocols.LoRaWan;

var phy = Convert.FromHexString("40F17DBE4900020001954378762B11FF0D");
foreach (var f in LoRaWanAnatomy.Describe(phy)) Console.WriteLine($"{f.Name,-11} {Convert.ToHexString(phy, f.Offset, f.Length),-10} {f.Value}");
var keys = LoRaWanSessionKeys.FromHex("44024241ED4CE9A68C6A8BC055233FD3", "EC925802AE430CA77FD3DD73CB2CC588");
var frame = LoRaWanPacket.Decode(phy);
Console.WriteLine($"MIC valid: {frame.VerifyMic(keys.NwkSKey)}, payload: {System.Text.Encoding.ASCII.GetString(frame.DecryptPayload(keys))}");

## Join over-the-air, secara manual
Perangkat dan jaringan menurunkan session key yang sama dari AppKey, DevNonce, dan Join-Accept. `LoRaWanEndDevice` adalah lapisan MAC perangkat tanpa I/O.

In [ ]:
var appKey = LoRaWanKeys.Parse("2B7E151628AED2A6ABF7158809CF4F3C");
var node = new LoRaWanEndDevice(Eui64.Parse("70B3D57ED0000101"), default, appKey);
var joinRequest = LoRaWanPacket.Decode(node.CreateJoinRequest());
Console.WriteLine(joinRequest);
var accept = new LoRaWanJoinAccept(JoinNonce: 0x00A1B2, NetId: 0x13, DevAddr: DevAddr.Parse("26011BDA"));
node.HandleDownlink(accept.Encode(appKey));
var networkKeys = accept.DeriveSessionKeys(appKey, joinRequest.DevNonce);
Console.WriteLine($"joined as {node.DevAddr}; same keys on both sides: {node.SessionKeys!.NwkSKey.SequenceEqual(networkKeys.NwkSKey)}");
var uplink = LoRaWanPacket.Decode(node.CreateUplink(1, new CayenneLpp().AddTemperature(1, 28.4).AddHumidity(2, 71).ToArray()));
Console.WriteLine($"{uplink} → {string.Join(", ", CayenneLpp.Decode(uplink.DecryptPayload(networkKeys)))}");

## Anggaran airtime
Setiap langkah dari SF7 ke SF12 menggandakan waktu di udara: jangkauan bertambah, kapasitas berkurang. Di EU868, duty cycle 1 % mengubahnya menjadi waktu tunggu wajib.

In [ ]:
foreach (var sf in new[] { 7, 8, 9, 10, 11, 12 })
{
    var air = LoRaAirtime.Compute(12 + 13, sf);
    Console.WriteLine($"SF{sf,-2} {air.TotalMilliseconds,7:0.0} ms   then wait {air.TotalSeconds * 99,6:0.0} s at 1 %");
}

## Jaringan di memori
Dua gateway meneruskan lewat Semtech UDP ke network server; empat sensor join dan melapor. `HonorTimestamps = false` melewati jeda join 5 detik yang sebenarnya agar sel cepat selesai.

In [ ]:
using System.Net;
using IoTCom.Net.Transports;

var radio = new InMemoryDatagramNetwork();
var nsAddress = new IPEndPoint(IPAddress.Parse("10.0.0.1"), 1700);
await using var ns = LoRaWanNetworkServer.Create(o => o.UseInMemory(radio, nsAddress).Region = LoRaRegion.AS923Group2);
var simOptions = LoRaWanSimulatorOptions.Demo(nsAddress);
simOptions.GatewayTransportFactory = () => radio.Bind();
simOptions.HonorTimestamps = false;
await using var sim = new LoRaWanSimulator(simOptions);
foreach (var r in sim.Registrations) ns.AddDevice(r);

var reports = new List<LoRaWanUplink>();
ns.UplinkReceived += (_, u) => { lock (reports) reports.Add(u); };
await ns.StartAsync();
await sim.StartAsync();
while (reports.Count < 4) await Task.Delay(100);
lock (reports)
    foreach (var u in reports)
        Console.WriteLine($"{u.Device.Name,-11} {u.DataRate,-9} via {u.Gateways.Count} gw, best SNR {u.Gateways[0].Snr,5:0.0}  {LoRaWanSimulator.DescribePayload(u.FPort, u.Payload)}");

## Downlink
Perangkat Class A hanya mendengar sesaat setelah mengirim, jadi downlink menunggu di antrean sampai uplink berikutnya. FPort 10 memberi tahu sensor simulasi seberapa sering melapor.

In [ ]:
var weather = ns.Devices.First(d => d.Name == "weather-01");
var weatherNode = sim.Devices.First(d => d.Name == "weather-01");
ns.EnqueueDownlink(weather.DevEui, 10, [0x00, 0x3C]);
sim.TriggerUplink("weather-01");
while (weatherNode.Interval != TimeSpan.FromSeconds(60)) await Task.Delay(100);
Console.WriteLine($"weather-01 now reports every {weatherNode.Interval.TotalSeconds} s ({weather.DownlinkCount} downlinks)");

## Lebih lanjut
Demo Galeri *Monitor jaringan LoRaWAN* menaruh gateway dan sensor di peta, lengkap dengan setiap uplink dan jendela RX-nya. `iotcom lorawan server --sim` menjalankan jaringan yang sama di terminal, dan `iotcom lorawan simulate` menggerakkan ChirpStack atau The Things Stack. Lihat `docs/id/protocols/lorawan.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*